# Depth orientation check for suspect test segments

Which depth direction is correct for eight test segments?
- Looks reversed: `20260814140748`, `20250703034159`, `20260723112922`
- Second request: `20260922073234`, `20260922161631`, `20260925085345`
- Off-centre or odd: `20260921094413`, `20260928000002`

**Reversed = the layer order flipped.** `assemble_test_segments.py` samples 28 layers at offsets (i − 13.5) along the mesh normal, using nearest-voxel sampling. Rendering with the normal flipped therefore gives exactly `zarr[::-1]`, so the reversed volume is built by flipping the fresh render. Everything else is shared between the two directions:
- the same footprint mask;
- the same normalisation stats (whole-volume masked stats don't change when z is flipped).

Surface supervision is regenerated for both directions with identical parameters.

**Why this counts as proof.** The test segments have no ink labels, so the model's output on them can't be scored directly. Instead:
1. **Calibration.** Take the native-96 holdout model (`36_holdout_native96`, fast eval: one surface-relative pass) and run it on the two holdout segments, 0841 and 0009B, in both directions. The model never trained on them, their correct direction is known, and they have ink labels. The supervised metrics (pAUC@1%FPR, recall@1% and @5% FPR) show how strongly the direction matters.
2. **Choosing the test statistics.** The candidate label-free statistics are fixed in advance: two from the model's predictions and three from the volume alone. A statistic counts as *validated* only if it favours the correct direction on both holdouts.
3. **Verdict.** Each test segment gets a direction only when every validated statistic agrees; otherwise it is `inconclusive`. Side-by-side fast figures (normal | reversed | composite) are saved to `output/orientation/figures/` for a visual check.

In [1]:
# settings
import json
import os
import sys
from pathlib import Path

REPO = Path("/vesuvius")
os.chdir(REPO)
sys.path.insert(0, str(REPO))

TEST_IDS = [
    20260814140748, 20250703034159, 20260723112922,   # flagged "looks reversed"
    20260922073234, 20260922161631, 20260925085345,   # second request
    20260921094413, 20260928000002,                   # flagged "off-centre or odd"
]
# held out from this model's training, known orientation, ink labels available
CALIB_IDS = [20260221022814, 20250919125754]
CALIB_NAMES = {20260221022814: "pherc0841", 20250919125754: "pherc0009b"}
DIRECTIONS = ("normal", "reversed")

EXP_NAME = "36_holdout_native96"
RUN_CONFIG_PATH = REPO / "runs_archs36" / EXP_NAME / "config.json"
MODEL_PATH = REPO / "models" / "archs36" / "holdout_native96" / "final_best_character.pth"

ZARR_DIR = REPO / "ves_zarrs2"
ORIENT = REPO / "output" / "orientation"
REVERSED_ZARR_DIR = REPO / "_ves_tmp" / "orientation_zarrs"   # git-ignored; flipped copies only
LABEL_DIR = REPO / "dilated_inklabels"                        # the run's inklabel_dir
EVAL_STRIDE = 32
MASK_CROP_MARGIN = 32
SURFACE_Z = (4, 28)                                           # same window as test_inference.ipynb
FORCE_SURFACE = True                                          # regenerate labels older than their zarr

with open(RUN_CONFIG_PATH, encoding="utf-8") as handle:
    RUN_CONFIG = json.load(handle)
_RUN_DATA, _RUN_MODEL = RUN_CONFIG["data"], RUN_CONFIG["model"]
assert int(_RUN_DATA["context_size"]) == 96 and int(_RUN_DATA["context_downsample"]) == 1
assert _RUN_DATA["surface_relative_depth_window"] and int(_RUN_DATA["depth"]) == 8
assert {"pherc0841", "pherc0009b"} <= set(_RUN_DATA["holdout_domains"]), "calibration segments must be held out"
for d in ("masks/reversed", "surface_labels/normal", "surface_labels/reversed", "preds", "figures", "review"):
    (ORIENT / d).mkdir(parents=True, exist_ok=True)
REVERSED_ZARR_DIR.mkdir(parents=True, exist_ok=True)
print(EXP_NAME, MODEL_PATH.name, "ctx", _RUN_DATA["context_size"], "holdouts", _RUN_DATA["holdout_domains"])

36_holdout_native96 final_best_character.pth ctx 96 holdouts ['pherc0841', 'pherc0009b']


In [2]:
# paths per (segment, direction); build reversed volumes and force surface supervision for both
import shutil
import subprocess

import cv2
import numpy as np
import zarr

from utils.dataloader import _load_unified_cache
from utils.norm import compute_norm


def zarr_path(sid, direction):
    return (ZARR_DIR if direction == "normal" else REVERSED_ZARR_DIR) / f"{sid}.zarr"


def mask_dir(sid, direction):
    return REPO / "masks" if direction == "normal" else ORIENT / "masks" / "reversed"


def surface_root(sid, direction):
    # test segments get their canonical labels forced; tracked holdout labels are left untouched
    if direction == "normal" and sid in TEST_IDS:
        return REPO / "surface_labels"
    return ORIENT / "surface_labels" / direction


def _mtime(path):
    return os.path.getmtime(path) if os.path.exists(path) else 0.0


def is_rendered(sid):
    # the assembler writes the mask after the last layer
    return zarr_path(sid, "normal").is_dir() and (REPO / "masks" / f"{sid}.png").is_file()


def ensure_reversed(sid):
    """flip the normal render in depth (== rendering with the normal inverted) and share its mask."""
    source = zarr.open(str(zarr_path(sid, "normal")), mode="r")
    target_path = zarr_path(sid, "reversed")
    if target_path.is_dir() and _mtime(target_path / ".zarray") >= _mtime(zarr_path(sid, "normal") / ".zarray"):
        return
    shutil.rmtree(target_path, ignore_errors=True)
    target = zarr.open(str(target_path), mode="w", shape=source.shape, chunks=source.chunks,
                       dtype=source.dtype, compressor=None)
    band = int(source.chunks[1]) * 16
    for y0 in range(0, source.shape[1], band):
        target[:, y0:y0 + band] = np.asarray(source[:, y0:y0 + band])[::-1]
    shutil.copyfile(REPO / "masks" / f"{sid}.png", mask_dir(sid, "reversed") / f"{sid}.png")
    print(f"[reverse] {sid} -> {target_path} {source.shape}")


def ensure_surface(sid, direction):
    out = surface_root(sid, direction) / str(sid)
    stale = _mtime(out / "depth.npy") < _mtime(zarr_path(sid, direction) / ".zarray")
    if (out / "depth.npy").is_file() and not (FORCE_SURFACE and stale):
        return
    print(f"[surface] {sid} {direction} -> {out}", flush=True)
    subprocess.run([
        sys.executable, str(REPO / "generate_surface_supervision.py"),
        "--scroll-id", str(sid), "--z-start", str(SURFACE_Z[0]), "--z-end", str(SURFACE_Z[1]),
        "--zarr-dir", str(zarr_path(sid, direction).parent),
        "--mask-dir", str(mask_dir(sid, direction)),
        "--output-dir", str(surface_root(sid, direction)),
        "--review-dir", str(ORIENT / "review" / direction),
    ], cwd=REPO, check=True, stdout=subprocess.DEVNULL)


def ensure_norm(sid):
    # the test renders are fresh, so their cached stats are recomputed once per render
    stats = _load_unified_cache().get(str(sid))
    marker = ORIENT / f".norm_{sid}"
    if stats and (sid not in TEST_IDS or _mtime(marker) >= _mtime(zarr_path(sid, "normal") / ".zarray")):
        return
    compute_norm(sid, str(ZARR_DIR))
    marker.touch()


READY_TEST = [sid for sid in TEST_IDS if is_rendered(sid)]
PENDING = [sid for sid in TEST_IDS if sid not in READY_TEST]
READY_IDS = CALIB_IDS + READY_TEST
assert all(is_rendered(sid) for sid in CALIB_IDS)
for sid in READY_IDS:
    ensure_reversed(sid)
    ensure_norm(sid)
    for direction in DIRECTIONS:
        ensure_surface(sid, direction)
print(f"ready: {len(READY_IDS)} segments x {len(DIRECTIONS)} directions; still rendering (rerun later): {PENDING}")

[reverse] 20260221022814 -> /vesuvius/_ves_tmp/orientation_zarrs/20260221022814.zarr (28, 3760, 4900)
[surface] 20260221022814 normal -> /vesuvius/output/orientation/surface_labels/normal/20260221022814
[surface] 20260221022814 reversed -> /vesuvius/output/orientation/surface_labels/reversed/20260221022814
[reverse] 20250919125754 -> /vesuvius/_ves_tmp/orientation_zarrs/20250919125754.zarr (28, 6497, 5260)
[surface] 20250919125754 normal -> /vesuvius/output/orientation/surface_labels/normal/20250919125754


KeyboardInterrupt: 

In [ ]:
# load the native-96 holdout model exactly as trained (full run config replayed, strict load)
import types

import torch

from utils.config import Config, ScrollConfig
from utils.model import create_model
from utils import visualizer as visualizer_module

TBV = visualizer_module.TensorboardVisualizer
_AUXILIARY_PREFIXES = ("supcon_head.", "domain_head.")


def build_config():
    config = Config()
    for name, value in _RUN_DATA.items():
        if name == "scrolls":
            config.data.scrolls = [ScrollConfig(**item) for item in value]
        elif name != "probe_rois" and hasattr(config.data, name):
            setattr(config.data, name, value)
    for name, value in _RUN_MODEL.items():
        if hasattr(config.model, name):
            setattr(config.model, name, value)
    config.data.eval_stride = EVAL_STRIDE
    config.model.compile_model = False
    config.tra.supcon = False
    config.tra.dann = False
    config.device = "cuda" if torch.cuda.is_available() else "cpu"
    return config


C = build_config()
MODEL, _params = create_model(C)
_state = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
_state = _state.get("state_dict", _state)
_state = {k.removeprefix("module.").removeprefix("_orig_mod."): v for k, v in _state.items()}
MODEL.load_state_dict({k: v for k, v in _state.items() if not k.startswith(_AUXILIARY_PREFIXES)}, strict=True)
MODEL.eval()
TILE = int(C.data.tile_size)
SUBTILE = int(C.model.multitile_subtile)
print(f"loaded {MODEL_PATH} params={_params:,} device={C.device} tile={TILE} subtile={SUBTILE}")

In [ ]:
# model-free evidence: layer-intensity profile vs the training segments, and surface-detector quality
import matplotlib.pyplot as plt

import campaign_archs_33 as campaign33
from utils.config import DEFAULT_TEST_SCROLL_IDS

HOLDOUT_IDS = set(CALIB_IDS) | {20251226000000}
REFERENCE_IDS = [int(s) for s in campaign33.PRETRAIN_SCROLL_IDS
                 if int(s) not in {int(t) for t in DEFAULT_TEST_SCROLL_IDS} | HOLDOUT_IDS
                 and is_rendered(int(s))]


def layer_profile(path, mask_path, step=8):
    vol = zarr.open(str(path), mode="r")
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (vol.shape[2], vol.shape[1]), interpolation=cv2.INTER_NEAREST)[::step, ::step] > 0
    sub = np.asarray(vol[:, ::step, ::step]).astype(np.float32)
    profile = np.array([layer[mask].mean() for layer in sub])
    return (profile - profile.mean()) / (profile.std() + 1e-9)


if "REFERENCE" not in globals():
    reference_profiles = {sid: layer_profile(zarr_path(sid, "normal"), REPO / "masks" / f"{sid}.png")
                          for sid in REFERENCE_IDS}
    REFERENCE = np.mean(list(reference_profiles.values()), axis=0)


def model_free_stats(sid, direction):
    profile = layer_profile(zarr_path(sid, direction), mask_dir(sid, direction) / f"{sid}.png")
    surface = surface_root(sid, direction) / str(sid)
    meta = json.loads((surface / "metadata.json").read_text())
    confidence = np.asarray(np.load(surface / "confidence.npy", mmap_mode="r")[::4, ::4])
    return {
        "profile_corr": float(np.corrcoef(profile, REFERENCE)[0, 1]),
        "surface_valid": float(meta["valid_fraction_inside_mask"]),
        "surface_conf": float(confidence[confidence > 0].mean() / 255),
    }, profile


MODEL_FREE, PROFILES = {}, {}
for sid in READY_IDS:
    for direction in DIRECTIONS:
        MODEL_FREE[sid, direction], PROFILES[sid, direction] = model_free_stats(sid, direction)

fig, axes = plt.subplots(1, len(READY_IDS), figsize=(3.2 * len(READY_IDS), 3), sharey=True, squeeze=False)
for ax, sid in zip(axes[0], READY_IDS):
    ax.plot(REFERENCE, "k-", lw=2, label=f"training mean (n={len(REFERENCE_IDS)})")
    ax.plot(PROFILES[sid, "normal"], "b-", label="normal")
    ax.plot(PROFILES[sid, "reversed"], "r--", label="reversed")
    ax.set_title(f"{CALIB_NAMES.get(sid, sid)}", fontsize=9)
    ax.set_xlabel("layer")
axes[0, 0].set_ylabel("z-scored mean intensity")
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
fig.savefig(ORIENT / "figures" / "layer_profiles.png", dpi=110)
plt.show()

In [ ]:
# fast eval: one surface-relative pass per (segment, direction); maps cached until the labels change
import time


def mask_bbox(mask, margin=MASK_CROP_MARGIN, align=None):
    align = align or TILE
    ys, xs = np.where(mask)
    y0 = max(0, (int(ys.min()) - margin) // align * align)
    x0 = max(0, (int(xs.min()) - margin) // align * align)
    y1 = min(mask.shape[0], (int(ys.max()) + 1 + margin + align - 1) // align * align)
    x1 = min(mask.shape[1], (int(xs.max()) + 1 + margin + align - 1) // align * align)
    return y0, y1, x0, x1


def full_mask(sid):
    # one footprint for both directions and for the metrics
    return cv2.imread(str(REPO / "masks" / f"{sid}.png"), cv2.IMREAD_GRAYSCALE) > 0


def predict(sid, direction):
    cache = ORIENT / "preds" / direction / f"{sid}.npy"
    if cache.is_file() and _mtime(cache) >= _mtime(surface_root(sid, direction) / str(sid) / "depth.npy"):
        return np.load(cache)
    started = time.time()
    mask_all = full_mask(sid)
    y0, y1, x0, x1 = mask_bbox(mask_all)
    vol = np.asarray(zarr.open(str(zarr_path(sid, direction)), mode="r")[:, y0:y1, x0:x1])
    mask = mask_all[y0:y1, x0:x1]
    surface = surface_root(sid, direction) / str(sid)
    depth_map = np.asarray(np.load(surface / "depth.npy", mmap_mode="r")[y0:y1, x0:x1])
    confidence = np.asarray(np.load(surface / "confidence.npy", mmap_mode="r")[y0:y1, x0:x1])
    stats = _load_unified_cache()[str(sid)]
    y_range, x_range = (0, y1 - y0), (0, x1 - x0)
    coords = TBV._gen_tile_coords(types.SimpleNamespace(c=C), (C.data.d_start, C.data.d_end),
                                  y_range, x_range, mask, z_step=C.data.depth)
    grouped = visualizer_module.group_by_depth(coords)
    assert len(grouped) == 1, "surface-relative inference expects one depth pass"
    offset, coords = next(iter(grouped.items()))
    with torch.no_grad():
        pred = visualizer_module.predict_tiles(
            C, MODEL, vol, mask, coords, y_range, x_range, C.data.d_start + offset,
            f"orient_{sid}_{direction}", stats["mean"], stats["std"], stats["min"], stats["max"],
            surface_depth_map=depth_map, surface_confidence_map=confidence, surface_depth_offset=0,
        ).astype(np.float32)
    cache.parent.mkdir(parents=True, exist_ok=True)
    np.save(cache, pred)
    print(f"[pred] {sid} {direction} {pred.shape} in {time.time() - started:.0f}s", flush=True)
    del vol
    torch.cuda.empty_cache()
    return pred


PREDS = {(sid, direction): predict(sid, direction) for sid in READY_IDS for direction in DIRECTIONS}

In [ ]:
# calibration: supervised metrics on the holdouts, then validate each label-free statistic
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score


def grid(image, pred_shape, bounds, interpolation):
    """full-frame image -> the prediction grid of the tile-aligned crop."""
    y0, y1, x0, x1 = bounds
    crop = image[y0:y1, x0:x1].astype(np.float32)
    return cv2.resize(crop, (pred_shape[1], pred_shape[0]), interpolation=interpolation)


def supervised(sid, direction):
    pred = PREDS[sid, direction]
    mask_all = full_mask(sid)
    bounds = mask_bbox(mask_all)
    label = cv2.imread(str(LABEL_DIR / f"{sid}.png"), cv2.IMREAD_GRAYSCALE)
    label = cv2.resize(label, (mask_all.shape[1], mask_all.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
    y = grid(label, pred.shape, bounds, cv2.INTER_AREA) >= 0.5
    valid = (grid(mask_all, pred.shape, bounds, cv2.INTER_AREA) >= 0.5) & np.isfinite(pred)
    y, s = y[valid], pred[valid]
    negatives = s[~y]
    return {
        "pAUC@1%": float(roc_auc_score(y, s, max_fpr=0.01)),
        "R@1%FPR": float((s[y] > np.quantile(negatives, 0.99)).mean()),
        "R@5%FPR": float((s[y] > np.quantile(negatives, 0.95)).mean()),
    }


def label_free(sid, direction):
    pred = PREDS[sid, direction]
    values = pred[np.isfinite(pred)]
    return {"ink_mass": float(values.mean()), "p99.5": float(np.percentile(values, 99.5)),
            **MODEL_FREE[sid, direction]}


STATS = ["ink_mass", "p99.5", "profile_corr", "surface_valid", "surface_conf"]
rows = []
for sid in CALIB_IDS:
    for direction in DIRECTIONS:
        rows.append({"segment": CALIB_NAMES[sid], "direction": direction,
                     **supervised(sid, direction), **label_free(sid, direction)})
CALIB_TABLE = pd.DataFrame(rows).set_index(["segment", "direction"])
display(CALIB_TABLE.round(4))

# validated: the sign of (normal - reversed) is the same on both holdouts and clears a 2% relative margin
VALIDATED = {}
for stat in STATS:
    deltas = [CALIB_TABLE.loc[(CALIB_NAMES[sid], "normal"), stat] - CALIB_TABLE.loc[(CALIB_NAMES[sid], "reversed"), stat]
              for sid in CALIB_IDS]
    scale = [max(abs(CALIB_TABLE.loc[(CALIB_NAMES[sid], d), stat]) for d in DIRECTIONS) for sid in CALIB_IDS]
    clear = all(abs(delta) > 0.02 * s for delta, s in zip(deltas, scale))
    if clear and len({np.sign(delta) for delta in deltas}) == 1:
        VALIDATED[stat] = int(np.sign(deltas[0]))
supervised_gap = {CALIB_NAMES[sid]: CALIB_TABLE.loc[(CALIB_NAMES[sid], "normal"), "pAUC@1%"]
                  - CALIB_TABLE.loc[(CALIB_NAMES[sid], "reversed"), "pAUC@1%"] for sid in CALIB_IDS}
print("pAUC@1% gain of the correct direction:", {k: round(v, 4) for k, v in supervised_gap.items()})
print("validated statistics (sign = +1 higher is correct):", VALIDATED or "none")

In [ ]:
# verdicts for the test segments + side-by-side fast figures (normal | reversed | composite)
if not VALIDATED:
    print("no statistic separated the holdout directions; every verdict below is inconclusive")

rows = []
for sid in READY_TEST:
    normal, reversed_ = label_free(sid, "normal"), label_free(sid, "reversed")
    votes = {stat: ("normal" if sign * (normal[stat] - reversed_[stat]) > 0 else "reversed")
             for stat, sign in VALIDATED.items()}
    agreed = set(votes.values())
    rows.append({
        "segment": sid,
        **{f"{stat} n/r": f"{normal[stat]:.3f} / {reversed_[stat]:.3f}" for stat in STATS},
        "votes": ", ".join(f"{stat}:{vote[0]}" for stat, vote in votes.items()),
        "verdict": agreed.pop() if len(agreed) == 1 else "inconclusive",
    })
VERDICTS = pd.DataFrame(rows).set_index("segment") if rows else pd.DataFrame()
display(VERDICTS)
VERDICTS.to_csv(ORIENT / "verdicts.csv")
CALIB_TABLE.to_csv(ORIENT / "calibration.csv")
if PENDING:
    print("not rendered yet, rerun from cell 3 later:", PENDING)

PRED_CMAP = plt.get_cmap("gray_r").copy()
PRED_CMAP.set_bad((0.78, 0.86, 0.95))
for sid in READY_IDS:
    mask_all = full_mask(sid)
    y0, y1, x0, x1 = mask_bbox(mask_all)
    vol = zarr.open(str(zarr_path(sid, "normal")), mode="r")
    composite = np.asarray(vol[10:18, y0:y1:4, x0:x1:4]).max(axis=0).astype(np.float32)
    composite = np.clip(composite, 0, np.percentile(composite[composite > 0], 99.5) if (composite > 0).any() else 1)
    height, width = composite.shape
    fig, axes = plt.subplots(1, 3, figsize=(15, 15 * height / (3 * width) + 0.8))
    for ax, direction in zip(axes[:2], DIRECTIONS):
        ax.imshow(np.ma.masked_invalid(PREDS[sid, direction]), cmap=PRED_CMAP, vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(direction)
    axes[2].imshow(composite, cmap="gray")
    axes[2].set_title("composite (normal, layers 10-18)")
    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
    verdict = VERDICTS.loc[sid, "verdict"] if sid in VERDICTS.index else "calibration: normal is correct"
    fig.suptitle(f"{CALIB_NAMES.get(sid, sid)}  -  {verdict}")
    plt.tight_layout()
    fig.savefig(ORIENT / "figures" / f"{sid}_normal_vs_reversed.jpg", dpi=90)
    plt.show()
    plt.close(fig)